In [0]:
# Databricks notebook: 00_Day4_PreFlight_Check
print("=" * 60)
print("PRE-FLIGHT CHECK — Day 4 Prerequisites")
print("=" * 60)

CATALOG = "insurance_dev"   # change to "workspace" if you used Fix B from earlier

checks = {
    "Catalog exists": f"SHOW CATALOGS",
    "Schemas exist": f"SHOW SCHEMAS IN {CATALOG}",
}

# 1. Catalog check
cats = [r.catalog for r in spark.sql("SHOW CATALOGS").collect()]
print(f"✅ Catalog '{CATALOG}' found" if CATALOG in cats else f"❌ Catalog '{CATALOG}' MISSING — go back to Day 1 setup")

# 2. Schema check
schemas = [r.databaseName for r in spark.sql(f"SHOW SCHEMAS IN {CATALOG}").collect()]
for s in ["bronze", "silver", "gold", "raw_files"]:
    status = "✅" if s in schemas else "❌ MISSING"
    print(f"{status} Schema: {CATALOG}.{s}")

# 3. Bronze tables from Day 1/3 check
print("\n--- Checking Bronze Tables ---")
expected_tables = ["customers", "agents", "policies", "premium_payments", "claims", "underwriting_risk"]
for t in expected_tables:
    try:
        cnt = spark.table(f"{CATALOG}.bronze.{t}").count()
        print(f"✅ {CATALOG}.bronze.{t:20s} → {cnt:,} rows")
    except Exception:
        print(f"❌ {CATALOG}.bronze.{t:20s} → NOT FOUND (re-run Day 1 load script)")

# 4. Volume check (needed for streaming/DLT source)
print("\n--- Checking Volumes ---")
try:
    vols = [r.volume_name for r in spark.sql(f"SHOW VOLUMES IN {CATALOG}.raw_files").collect()]
    print(f"Volumes found: {vols}")
    if "claims_stream" not in vols:
        print("⚠️  'claims_stream' volume missing — creating it now for Day 4...")
        spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.raw_files.claims_stream")
        print("✅ Created.")
except Exception as e:
    print(f"❌ Could not check volumes: {e}")

print("\n" + "=" * 60)
print("Pre-flight check complete. Fix any ❌ before continuing.")
print("=" * 60)

PRE-FLIGHT CHECK — Day 4 Prerequisites
✅ Catalog 'insurance_dev' found
✅ Schema: insurance_dev.bronze
✅ Schema: insurance_dev.silver
✅ Schema: insurance_dev.gold
✅ Schema: insurance_dev.raw_files

--- Checking Bronze Tables ---
✅ insurance_dev.bronze.customers            → 50,000 rows
✅ insurance_dev.bronze.agents               → 1,000 rows
✅ insurance_dev.bronze.policies             → 200,000 rows
✅ insurance_dev.bronze.premium_payments     → 500,000 rows
✅ insurance_dev.bronze.claims               → 100,000 rows
✅ insurance_dev.bronze.underwriting_risk    → 200,000 rows

--- Checking Volumes ---
Volumes found: ['claims_stream', 'landing']

Pre-flight check complete. Fix any ❌ before continuing.


Step 1 — Data Quality + manual Bronze/Silver/Gold

Run exactly as written in Day 4 Part A, just qualify table names with insurance_dev.silver. / insurance_dev.gold. etc.

In [0]:

CATALOG = "insurance_dev"

# Read the bronze claims table you already loaded

df_claims_raw = spark.table(f"{CATALOG}.bronze.claims")

print(f"Total raw claims: {df_claims_raw.count()}")
df_claims_raw.limit(25).display()

# Quick look at schema - confirms data types are what we expect
df_claims_raw.printSchema()

CATALOG = "insurance_dev"

# Read the bronze policies table you already loaded

df_policies_raw = spark.table(f"{CATALOG}.bronze.policies")

print(f"Total raw policies: {df_policies_raw.count():,}")
df_policies_raw.limit(25).display()

# Quick look at schema - confirms data types are what we expect
df_policies_raw.printSchema()

Total raw claims: 100000


claim_id,policy_id,claim_date,claim_amount,claim_type,claim_status,fraud_flag
CLM0000001,POL0017679,2023-08-27,69816.37,DeathBenefit,Approved,false
CLM0000002,POL0127286,2022-10-12,448022.87,Accident,Filed,false
CLM0000003,POL0059375,2022-10-03,184812.35,DeathBenefit,Approved,false
CLM0000004,POL0127434,2020-07-16,19076.98,Surgery,UnderReview,false
CLM0000005,POL0136627,2023-10-30,466938.05,Theft,Approved,false
CLM0000006,POL0187410,2023-07-16,346433.45,DeathBenefit,UnderReview,false
CLM0000007,POL0047021,2022-04-29,479775.74,Accident,UnderReview,false
CLM0000008,POL0014380,2024-05-20,206279.7,Theft,UnderReview,false
CLM0000009,POL0076835,2020-08-16,466619.05,Theft,Approved,false
CLM0000010,POL0153253,2023-10-14,242172.56,Theft,Filed,false


root
 |-- claim_id: string (nullable = true)
 |-- policy_id: string (nullable = true)
 |-- claim_date: date (nullable = true)
 |-- claim_amount: double (nullable = true)
 |-- claim_type: string (nullable = true)
 |-- claim_status: string (nullable = true)
 |-- fraud_flag: boolean (nullable = true)

Total raw policies: 200,000


policy_id,customer_id,agent_id,policy_type,start_date,end_date,premium_amount,status
POL0000001,CUST016007,AGT00771,Health,2020-07-16,2028-11-14,57328.71,Active
POL0000002,CUST009710,AGT00541,Life,2018-08-19,2027-08-19,67244.82,Expired
POL0000003,CUST005598,AGT00461,Auto,2019-02-23,2027-08-30,84496.96,Active
POL0000004,CUST028183,AGT00565,Health,2021-01-06,2029-11-22,10457.83,Cancelled
POL0000005,CUST047621,AGT00636,Life,2018-03-04,2028-05-22,32143.09,Active
POL0000006,CUST006805,AGT00631,Health,2023-03-16,2024-09-16,87204.83,Active
POL0000007,CUST011932,AGT00009,Auto,2018-07-09,2030-06-23,95335.46,Lapsed
POL0000008,CUST045131,AGT00787,Health,2018-08-16,2024-12-01,71242.23,Active
POL0000009,CUST043040,AGT00838,Health,2022-09-06,2029-08-11,85354.32,Active
POL0000010,CUST017610,AGT00056,Life,2020-11-09,2029-10-13,53873.41,Active


root
 |-- policy_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- agent_id: string (nullable = true)
 |-- policy_type: string (nullable = true)
 |-- start_date: date (nullable = true)
 |-- end_date: date (nullable = true)
 |-- premium_amount: double (nullable = true)
 |-- status: string (nullable = true)



Step 1.3 — Define Your Data Quality Rules (Explicitly, One at a Time)

Rather than one giant filter, we'll build rules individually so you can see exactly which rule catches which bad row — critical for real-world debugging.

In [0]:
from pyspark.sql.functions import col, when, lit, current_timestamp

#Rule Definitions  - each rule is one Business requirement#

#Rule -1: claim_amount nmust be positive
rule1_bad = df_claims_raw.filter((col("claim_amount") <=0) | col("claim_amount").isNull())
print(f"Rule 1 Violations(claim_amount <=0 or null): {rule1_bad.count():,}")

#Rule -2: claim_id must not be null
rule2_bad = df_claims_raw.filter(col("claim_id").isNull())
print(f"Rule 2 viloations (null claim_id):{rule2_bad.count()}")

#Rule 3: claim_type must not be null
rule3_bad = df_claims_raw.filter(col("claim_type").isNull())
print(f"Rule 3 viloations (null claim_type):{rule3_bad.count()}")
display(rule3_bad.select("claim_id","policy_id", "claim_type"))

#Rule 4: policy_id must exists in policies table(Referential integrity)
df_policies = spark.table(f"{CATALOG}.bronze.policies")
valid_policies_ids = df_policies.select("policy_id").distinct()

rule4_bad = df_claims_raw.join(valid_policies_ids, "policy_id", "left_anti")
print(f"Rule 4 violations (orphan policy_id, no matching policy):{rule4_bad.count()}")
display(rule4_bad.select("claim_id","policy_id", "claim_date"))

#Rule 5: claim_date must be after policy start_date
rule5_bad = df_claims_raw.join(df_policies_raw.select("policy_id", "start_date"), on="policy_id", how="inner").filter(col("claim_date") < col("start_date"))
print(f"Rule 5 viloations (claim_date < start_date):{rule5_bad.count()}")


#Rule 6: claim_id must be unique (no duplicates)
from pyspark.sql.window import Window
from pyspark.sql.functions import count as _count

dup_window = Window.partitionBy("claim_id")
df_with_dup_flag = df_claims_raw.withColumn("dup_count", _count("claim_id").over(dup_window))
rule6_bad = df_with_dup_flag.filter(col("dup_count") >1)
print(f"Rule 6 ciolations (duplciate claim_id): {rule6_bad.count()}")
display(rule6_bad.select("claim_id","policy_id", "dup_count"))


Rule 1 Violations(claim_amount <=0 or null): 0
Rule 2 viloations (null claim_id):0
Rule 3 viloations (null claim_type):0


claim_id,policy_id,claim_type


Rule 4 violations (orphan policy_id, no matching policy):0


claim_id,policy_id,claim_date


Rule 5 viloations (claim_date < start_date):26916
Rule 6 ciolations (duplciate claim_id): 0


claim_id,policy_id,dup_count


Step 1.4 — Build the Quarantine + Clean Split (Combine All Rules)

In [0]:
# ============================================================
# COMBINE ALL RULES INTO ONE PASS/FAIL DECISION PER ROW
# ============================================================

df_tagged = (df_with_dup_flag
    .withColumn("fails_amount_rule", (col("claim_amount")<=0) | col("claim_amount").isNull())
    .withColumn("fails_id_rule",  col("claim_id").isNull())
    .withColumn("fails_type_rule", col("claim_type").isNull())
    .withColumn("fails_duplicate_rule" , col("dup_count")>1)
    )                                          

# A row is BAD if it fails ANY rule
df_tagged = df_tagged.withColumn(
    "is_valid",
    ~(col("fails_amount_rule") | col("fails_id_rule") | col("fails_type_rule") | col("fails_duplicate_rule"))
)

# Also flag orphan policy_ids (join-based check, added separately)
orphan_ids = rule4_bad.select("claim_id").withColumnRenamed("claim_id", "orphan_claim_id")
df_tagged = df_tagged.join(orphan_ids, df_tagged.claim_id == orphan_ids.orphan_claim_id, "left") \
    .withColumn("fails_orphan_rule", col("orphan_claim_id").isNotNull()) \
    .drop("orphan_claim_id")

df_tagged = df_tagged.withColumn(
    "is_valid",
    col("is_valid") & ~col("fails_orphan_rule")
)

# Add audit metadata — WHEN was this quality check run
df_tagged = df_tagged.withColumn("quality_checked_at", current_timestamp())

# ============================================================
# SPLIT: Clean records go to Silver, bad records go to Quarantine
# ============================================================

df_clean = df_tagged.filter(col("is_valid") == True).drop(
    "dup_count", "fails_amount_rule", "fails_id_rule", "fails_type_rule",
    "fails_duplicate_rule", "fails_orphan_rule", "is_valid"
)

df_quarantine = df_tagged.filter(col("is_valid") == False)

print(f" Clean records (→ Silver):      {df_clean.count():,}")
print(f" Quarantined records (→ Audit): {df_quarantine.count():,}")

display(df_quarantine.select(
    "claim_id", "policy_id", "claim_amount", "claim_type",
    "fails_amount_rule", "fails_id_rule", "fails_type_rule",
    "fails_duplicate_rule", "fails_orphan_rule"
))

 Clean records (→ Silver):      100,000
 Quarantined records (→ Audit): 0


claim_id,policy_id,claim_amount,claim_type,fails_amount_rule,fails_id_rule,fails_type_rule,fails_duplicate_rule,fails_orphan_rule


Step 1.5 — Write to Silver and Quarantine Tables

In [0]:
# Write CLEAN data to Silver
df_clean.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{CATALOG}.silver.claims_clean")

# Write BAD data to a Quarantine table (also in silver schema, clearly named)
df_quarantine.write.format("delta").mode("overwrite")  \
    .saveAsTable(f"{CATALOG}.silver.claims_quarantine")

print("Written:")
print(f" {CATALOG}.silver.claims_clean ({spark.table(f'{CATALOG}.silver.claims_clean').count():,} rows)")
print(f" {CATALOG}.silver.claims_quarantine ({spark.table(f'{CATALOG}.silver.claims_quarantine').count():,}) rows")


Written:
 insurance_dev.silver.claims_clean (100,000 rows)
 insurance_dev.silver.claims_quarantine (0) rows


Verify in SQL:

In [0]:
%sql
SELECT 'clean' AS bucket, COUNT(*) As n FROM insurance_dev.silver.claims_clean
UNION ALL
SELECT 'quarantine' AS bucket, COUNT(*) As n FROM insurance_dev.silver.claims_quarantine;


bucket,n
clean,100000
quarantine,0


Step 1.6 — Build the Gold Aggregate Layer

In [0]:
from pyspark.sql.functions import sum as _sum, avg as _avg, count as _count2

df_gold = spark.sql (f"""
    SELECT 
        claim_type,
        claim_status,
        COUNT(*) AS claim_count,
        SUM(claim_amount) AS total_claim_amount,
        ROUND(AVG(claim_amount), 2) AS avg_claim_amount,
        SUM(CASE WHEN fraud_flag THEN 1 ELSE 0 END) AS fraud_claim_count,
        ROUND(SUM(CASE WHEN fraud_flag THEN claim_amount ELSE 0 END), 2) AS total_fraud_claim_amount   
    FROM {CATALOG}.silver.claims_clean
    GROUP BY claim_type, claim_status
    ORDER BY total_claim_amount DESC
                     """)

df_gold.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{CATALOG}.gold.claims_summary")

print("Written:")
display(spark.table(f"{CATALOG}.gold.claims_summary"))
                    


Written:


claim_type,claim_status,claim_count,total_claim_amount,avg_claim_amount,fraud_claim_count,total_fraud_claim_amount
NaturalDisaster,Approved,6738,1.6912326803499951E9,250999.21,335,8.754927733E7
DeathBenefit,Approved,6656,1.6796711062100024E9,252354.43,366,9.616997981E7
Accident,Approved,6638,1.655120082400002E9,249340.18,303,7.583432932E7
Theft,Approved,6632,1.6539119431100013E9,249383.59,294,6.990155969E7
Fire,Approved,6554,1.6432631903500006E9,250726.76,333,8.547170092E7
Surgery,Approved,6602,1.63911465808E9,248275.47,348,8.688437079E7
Theft,UnderReview,5030,1.268885957689995E9,252263.61,253,6.150693233E7
DeathBenefit,UnderReview,5072,1.261567965990003E9,248731.85,243,6.263020841E7
Surgery,UnderReview,5038,1.2568515193199973E9,249474.3,280,6.762201547E7
NaturalDisaster,UnderReview,4969,1.250904802550001E9,251741.76,272,6.556610887E7


A Loss Ratio Gold table (actuarial metric) 
 
 
Loss Ratio = Total Claims Paid ÷ Total Premium Collected. A ratio above 1.0 means InsureCo is paying out MORE than it collects for that policy type — a red flag for pricing teams.

In [0]:
df_loss_ratio = spark.sql (f"""
    WITH premium_totals AS (
        SELECT p.policy_type, SUM(p.premium_amount) AS total_premium
        FROM {CATALOG}.bronze.policies p
        GROUP BY p.policy_type
    ),
    claim_totals AS (
        SELECT pol.policy_type, SUM(c.claim_amount) AS total_claims
        FROM {CATALOG}.silver.claims_clean c
        JOIN {CATALOG}.bronze.policies pol ON c.policy_id = pol.policy_id
        GROUP BY pol.policy_type
    )
   SELECT
   pt.policy_type,
   pt.total_premium,
   COALESCE (ct.total_claims, 0) AS total_claims,
   ROUND(COALESCE(ct.total_claims, 0) / pt.total_premium , 3) AS loss_ratio
   FROM premium_totals pt
   LEFT JOIN claim_totals ct ON pt.policy_type = ct.policy_type
   ORDER BY loss_ratio DESC
 """)

df_loss_ratio.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{CATALOG}.gold.loss_ratio_by_policy_type")

display(df_loss_ratio)    


policy_type,total_premium,total_claims,loss_ratio
Auto,2.640840872750004E9,6.357234294460002E9,2.407
Home,2.6295037399799876E9,6.282556792229982E9,2.389
Life,2.6260453105199833E9,6.26912696432998E9,2.387
Health,2.6016459353700056E9,6.162741281999982E9,2.369


Step 1.7 — Full Verification of Step 1

In [0]:
%sql
-- Run all of these and confirm no errors
SELECT COUNT(*) FROM insurance_dev.silver.claims_clean;
SELECT COUNT(*) FROM insurance_dev.silver.claims_quarantine;
SELECT * FROM insurance_dev.gold.claims_summary;
SELECT * FROM insurance_dev.gold.loss_ratio_by_policy_type;

policy_type,total_premium,total_claims,loss_ratio
Auto,2.640840872750004E9,6.357234294460002E9,2.407
Home,2.6295037399799876E9,6.282556792229982E9,2.389
Life,2.6260453105199833E9,6.26912696432998E9,2.387
Health,2.6016459353700056E9,6.162741281999982E9,2.369
